Concurrency  

Tasks take turns executing.  
Task1 → Task2 → Task3 → Task1 → Task2

Parallelism  

Tasks run at the same time on different CPUs.  

CPU1 → Task1  
CPU2 → Task2  
CPU3 → Task3

Python GIL (Global Interpreter Lock)  

    Very important concept.

In CPython:  

Only ONE thread executes Python bytecode at a time.  

So even with multiple threads:  

CPU-bound tasks do not run in parallel.

In [ ]:
import threading

def task():
    for i in range(10_000_000):
        pass

threads = [threading.Thread(target=task) for _ in range(4)]

for t in threads:
    t.start()

for t in threads:
    t.join()
# Despite 4 threads, they do not run in parallel because of the GIL.

# When Threads Are Useful
# Threads work well for I/O-bound tasks.

# Example:
# API requests
# file downloads
# database calls
# waiting for disk/network
# Because while one thread waits, another can run.

In [9]:
import threading
import time

def download():
    print("Start")
    time.sleep(2)
    print("Done")

threads = [threading.Thread(target=download) for _ in range(3)]

start = time.perf_counter()
for t in threads:
    t.start()
for t in threads:
    t.join()
print(time.perf_counter() - start) 
# All downloads start together.

Start
Start
Start
DoneDone

Done
2.006327249997412


CPU-bound  
    Heavy computation.

Example:  
    encryption  
    image processing  
    ML training  
    Threads do not help much due to GIL.

In [ ]:
# Multiprocessing uses separate processes.
# Each process has its own Python interpreter and GIL.

In [ ]:
from multiprocessing import Process

def task():
    for i in range(10_000_000):
        pass

processes = [Process(target=task) for _ in range(4)]

for p in processes:
    p.start()

for p in processes:
    p.join()

| Feature      | Threads       | Processes |
| ------------ | ------------- | --------- |
| Memory       | Shared        | Separate  |
| Speed        | Fast creation | Slower    |
| CPU parallel | ❌             | ✔         |
| Best for     | I/O tasks     | CPU tasks |


## ✅ What is Thread Pool?

A pool of **pre-created threads** used to execute tasks concurrently.

* Avoids creating/destroying threads repeatedly
* Uses a **task queue + worker threads**

---

## ⚙️ Python Implementation

### Using `ThreadPoolExecutor`

```python
from concurrent.futures import ThreadPoolExecutor

def task(n):
    return f"Task {n} done"

with ThreadPoolExecutor(max_workers=3) as executor:
    results = list(executor.map(task, range(5)))

print(results)
```

---

## 🔑 Key Concepts

* `max_workers` → number of threads
* `submit(fn, args)` → submit single task
* `map(fn, iterable)` → apply function to multiple inputs
* Tasks wait in **queue** if all threads busy

---

## 🔥 Example (submit)

```python
from concurrent.futures import ThreadPoolExecutor

def task(n):
    print(f"Running {n}")

with ThreadPoolExecutor(max_workers=2) as executor:
    for i in range(5):
        executor.submit(task, i)
```

---

## ⚠️ When to Use

✔ I/O-bound tasks:

* API calls
* File reading
* DB queries

❌ Not for CPU-bound tasks (due to GIL)

---

## 🚀 Interview One-Liner

> ThreadPoolExecutor manages a pool of worker threads to execute tasks concurrently, improving performance for I/O-bound workloads.

---

## ⚠️ Important Notes

* Uses **threads (not processes)**
* Limited by **GIL** for CPU work
* Automatically handles thread lifecycle

---

## 🧠 Alternative for CPU Tasks

Use:

```python
from concurrent.futures import ProcessPoolExecutor
```

---

## 📌 Summary

```
Thread Pool = Threads + Task Queue
Best for = I/O-bound tasks
Avoid for = CPU-heavy tasks
```


## what it is

* pool of worker **processes** (not threads)
* each process has its own Python interpreter + memory
* avoids GIL → true parallel execution

---

## when to use

* CPU heavy work:

  * data processing
  * number crunching
  * image/video ops

not ideal for:

* simple I/O tasks (threads/async cheaper)

---

## basic usage

```python
from concurrent.futures import ProcessPoolExecutor

def square(x):
    return x * x

with ProcessPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(square, range(10)))

print(results)
```

---

## submit example

```python
from concurrent.futures import ProcessPoolExecutor

def work(x):
    return x + 10

with ProcessPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(work, i) for i in range(5)]
    results = [f.result() for f in futures]

print(results)
```

---

## key points

* `max_workers` → number of processes
* uses multiple CPU cores
* no shared memory (data is pickled and sent)
* higher overhead than threads

---

## important gotchas

* functions must be **picklable** (no lambdas, no local funcs)
* on Windows/macOS → wrap code in:

```python
if __name__ == "__main__":
    ...
```

* startup cost is higher than threads

---

## thread vs process (quick view)

| type    | best for | GIL | overhead |
| ------- | -------- | --- | -------- |
| thread  | I/O work | yes | low      |
| process | CPU work | no  | higher   |

---

## one-liner

process pool = multiple processes running tasks in parallel to fully use CPU cores

---

## mental model

* thread pool → lightweight, shared memory
* process pool → heavy, isolated but parallel


# Race Condition in Python (GIL + Locks)

## code in question

```python
counter += 1
```

---

## what actually happens (internally)

not a single step:

1. read counter
2. load 1
3. add
4. write back

👉 multiple steps → not atomic

---

## where problem occurs

```text
Thread A → read (0)
Thread B → read (0)
Thread B → write 1
Thread A → write 1   ❌ lost update
```

---

## why it “looks safe”

* GIL → only one thread runs at a time
* switching doesn’t always happen mid-operation
* tight loops → fewer context switches

👉 so result often looks correct

---

## important truth

* race condition **exists**
* not guaranteed safe
* depends on timing/scheduling

---

## forcing the issue

```python
temp = counter
time.sleep(0.00001)
counter = temp + 1
```

👉 now wrong results appear

---

# 🔒 locks (solution)

## what is a lock

* a mechanism to allow **only one thread** into critical section
* others wait until lock is released

---

## usage

```python
from threading import Lock

lock = Lock()

def inc():
    global counter
    with lock:
        counter += 1
```

---

## how it works

```text
Thread A → acquires lock → updates → releases
Thread B → waits → then runs
```

👉 prevents overlap → no lost updates

---

## terms

* critical section → shared data/code
* mutex/lock → mutual exclusion
* race condition → timing-dependent bug

---

## downsides of locks

* slower (threads wait)
* can cause **deadlock** if:

  * multiple locks
  * wrong ordering

---

## quick rules

* shared state → use lock
* read-only → safe
* no shared state → no issue

---

## alternativ


In [30]:
import threading
import time

counter = 0

def inc():
    global counter
    for _ in range(10000):
        temp = counter
        time.sleep(0.00001)  # force context switch
        counter = temp + 1

threads = []

for _ in range(2):
    t = threading.Thread(target=inc)
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print("Final counter:", counter)

Final counter: 10001
